# lstm_code

LSTM es el modelo de Deep Learning del estudio. La red escala las entradas, utiliza `dropout` y aplica parada temprana para reducir el sobreajuste. Una mayor complejidad no garantiza mejores resultados: aproximadamente 130 observaciones mensuales pueden ser insuficientes para que una red neuronal aprenda dependencias temporales profundas de forma robusta.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install tensorflow matplotlib pandas numpy scikit-learn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Silenciar logs de TensorFlow (no afecta a resultados)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.get_logger().setLevel("ERROR")

SEMILLA = 42
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)

# 1. CARGA DE DATOS Y CONSTRUCCIÓN DE LA SERIE DIFERENCIADA
RUTA_DATOS = "/content/drive/MyDrive/VIU/TFM/Data/DATASET_TFM_FINAL.csv"
TARGET = "Pernoctaciones"
MESES_TEST = 18    # misma ventana de test que los demás modelos
LOOK_BACK = 12     # meses de historia que ve la LSTM en cada paso (1 año)

df = pd.read_csv(RUTA_DATOS, parse_dates=["fecha"], index_col="fecha")
df = df.sort_index()
y_abs = df[TARGET].values.astype(float)

# Features de calendario concatenadas al estado LSTM (sin fuga: son conocidas de antemano para cualquier mes futuro)
CALENDAR_FEATURES = ["mes_sin", "mes_cos", "es_covid"]
cal = df[CALENDAR_FEATURES].values.astype(float)

# Diferencias de primer orden (igual que en XGBoost/RF/LR)
y_diff = np.diff(y_abs, prepend=np.nan)   # y_diff[t] = y_abs[t] - y_abs[t-1]


# 2. NORMALIZACIÓN (obligatoria en LSTM, ajustada solo sobre train)
n_total = len(y_abs)
idx_train_end = n_total - MESES_TEST

scaler_diff = MinMaxScaler(feature_range=(-1, 1))
scaler_cal = MinMaxScaler(feature_range=(-1, 1))

# Ajuste SOLO con datos de train para no filtrar información del test
scaler_diff.fit(y_diff[1:idx_train_end].reshape(-1, 1))
scaler_cal.fit(cal[:idx_train_end])

y_diff_scaled = np.full_like(y_diff, np.nan)
y_diff_scaled[1:] = scaler_diff.transform(y_diff[1:].reshape(-1, 1)).flatten()
cal_scaled = scaler_cal.transform(cal)

# 3. CONSTRUCCIÓN DE SECUENCIAS (ventana deslizante LOOK_BACK -> 1 paso)
def crear_secuencias(y_diff_s, cal_s, y_abs_ref, look_back):
    """Genera (X_seq, X_cal, y_diff_target, y_abs_ancla, y_abs_objetivo):
    - X_seq[i]:        ventana de look_back diferencias escaladas hasta t
    - X_cal[i]:        features de calendario del mes OBJETIVO (t+1), escaladas
    - y_diff_target:   diferencia escalada del mes objetivo (lo que aprende LSTM)
    - y_abs_ancla:     valor absoluto real en t (para reconstruir la predicción)
    - y_abs_objetivo:  valor absoluto real en t+1 (para evaluar el modelo)
    """
    X_seq, X_cal_out, y_diff_t, y_abs_anc, y_abs_obj = [], [], [], [], []
    for i in range(look_back, len(y_diff_s) - 1):
        if np.isnan(y_diff_s[i - look_back:i]).any():
            continue
        X_seq.append(y_diff_s[i - look_back:i])
        X_cal_out.append(cal_s[i + 1])           # calendario del mes que predicemos
        y_diff_t.append(y_diff_s[i + 1])         # diferencia que la LSTM aprende
        y_abs_anc.append(y_abs_ref[i])            # ancla para reconstruir absoluto
        y_abs_obj.append(y_abs_ref[i + 1])        # valor real a predecir
    return (np.array(X_seq, dtype=np.float32)[..., np.newaxis],
            np.array(X_cal_out, dtype=np.float32),
            np.array(y_diff_t, dtype=np.float32),
            np.array(y_abs_anc, dtype=np.float32),
            np.array(y_abs_obj, dtype=np.float32))


X_seq, X_cal_seq, y_diff_seq, y_anc_seq, y_obj_seq = crear_secuencias(
    y_diff_scaled, cal_scaled, y_abs, LOOK_BACK
)

# Split temporal: las últimas MESES_TEST secuencias son el test
X_seq_train,  X_seq_test  = X_seq[:-MESES_TEST],     X_seq[-MESES_TEST:]
X_cal_train,  X_cal_test  = X_cal_seq[:-MESES_TEST], X_cal_seq[-MESES_TEST:]
y_diff_train               = y_diff_seq[:-MESES_TEST]
y_anc_test                 = y_anc_seq[-MESES_TEST:]
y_obj_test                 = y_obj_seq[-MESES_TEST:]


# 4. BÚSQUEDA DE ARQUITECTURA LSTM
#    (equivalente a la doble optimización RandomizedSearch -> GridSearch)
# FASE 1: Búsqueda amplia — evaluamos 5 configuraciones de profundidad y
#         unidades con Early Stopping, equivalente a RandomizedSearchCV.
# FASE 2: Afinado — reentrenamos la mejor con más épocas y paciencia mayor,
#         equivalente a GridSearchCV refinado.
print(">> Ejecutando Optimización de Arquitectura LSTM (búsqueda amplia + afinado)...")

CONFIGS_CANDIDATAS = [
    {"units_1": 64,  "units_2": None, "dropout": 0.2},
    {"units_1": 128, "units_2": None, "dropout": 0.2},
    {"units_1": 64,  "units_2": 32,   "dropout": 0.2},
    {"units_1": 128, "units_2": 64,   "dropout": 0.3},
    {"units_1": 64,  "units_2": 32,   "dropout": 0.3},
]


def construir_modelo(units_1, units_2, dropout, n_cal_features, look_back):
    """Arquitectura LSTM con rama de calendario concatenada antes de la capa
    densa. Combina el patrón secuencial de la serie con features de calendario
    conocidas de antemano para el mes a predecir."""
    # Rama LSTM (serie temporal diferenciada)
    entrada_seq = Input(shape=(look_back, 1), name="entrada_secuencia")
    if units_2 is not None:
        x = LSTM(units_1, return_sequences=True, name="lstm_1")(entrada_seq)
        x = Dropout(dropout, name="dropout_1")(x)
        x = LSTM(units_2, return_sequences=False, name="lstm_2")(x)
        x = Dropout(dropout, name="dropout_2")(x)
    else:
        x = LSTM(units_1, return_sequences=False, name="lstm_1")(entrada_seq)
        x = Dropout(dropout, name="dropout_1")(x)

    # Rama calendario (features del mes objetivo, conocidas de antemano)
    entrada_cal = Input(shape=(n_cal_features,), name="entrada_calendario")

    # Concatenación + capa de salida
    combinado = Concatenate(name="concatenar")([x, entrada_cal])
    salida = Dense(1, name="salida")(combinado)

    modelo = Model(inputs=[entrada_seq, entrada_cal], outputs=salida)
    modelo.compile(optimizer=Adam(learning_rate=1e-3), loss="mse")
    return modelo


early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

mejor_val_loss = np.inf
mejor_config = None

for cfg in CONFIGS_CANDIDATAS:
    modelo_prueba = construir_modelo(
        cfg["units_1"], cfg["units_2"], cfg["dropout"],
        n_cal_features=len(CALENDAR_FEATURES), look_back=LOOK_BACK,
    )
    historia = modelo_prueba.fit(
        [X_seq_train, X_cal_train], y_diff_train,
        epochs=100, batch_size=16, verbose=0,
        validation_split=0.15,
        callbacks=[early_stop],
    )
    val_loss_min = min(historia.history["val_loss"])
    print(f"   Config {cfg} -> val_loss={val_loss_min:.5f}")
    if val_loss_min < mejor_val_loss:
        mejor_val_loss = val_loss_min
        mejor_config = cfg

print(f"\nMejor configuración: {mejor_config} (val_loss={mejor_val_loss:.5f})")

# FASE 2: Afinado — reentrenamos con más épocas y mayor paciencia
print(">> Afinado: reentrenando mejor configuración con mayor paciencia...")
early_stop_fino = EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5)

modelo_final = construir_modelo(
    mejor_config["units_1"], mejor_config["units_2"], mejor_config["dropout"],
    n_cal_features=len(CALENDAR_FEATURES), look_back=LOOK_BACK,
)
historia_final = modelo_final.fit(
    [X_seq_train, X_cal_train], y_diff_train,
    epochs=300, batch_size=16, verbose=0,
    validation_split=0.15,
    callbacks=[early_stop_fino, reduce_lr],
)
print(f"Entrenamiento finalizado en época {len(historia_final.history['loss'])}\n")

# 5. PREDICCIÓN Y RECONSTRUCCIÓN MATEMÁTICA
pred_diff_scaled = modelo_final.predict(
    [X_seq_test, X_cal_test], verbose=0
).flatten()

# Desnormalizar diferencias predichas al espacio original
pred_diff = scaler_diff.inverse_transform(
    pred_diff_scaled.reshape(-1, 1)
).flatten()

# Reconstrucción: absoluto(t+1) = valor_real(t) + diferencia_predicha(t->t+1)
pred_abs  = y_anc_test + pred_diff
y_test_abs = y_obj_test


# 6. MÉTRICAS DE EVALUACIÓN FINAL
mae  = mean_absolute_error(y_test_abs, pred_abs)
rmse = np.sqrt(mean_squared_error(y_test_abs, pred_abs))
mape = np.mean(np.abs((y_test_abs - pred_abs) / y_test_abs)) * 100
r2   = r2_score(y_test_abs, pred_abs)

arq_str = (
    f"LSTM({mejor_config['units_1']}"
    + (f"+{mejor_config['units_2']}" if mejor_config["units_2"] else "")
    + f", dropout={mejor_config['dropout']}, look_back={LOOK_BACK})"
)

print("\n" + "="*40)
print("         MÉTRICAS DEEP LEARNING (LSTM)")
print("="*40)
print(f"MAE:  {mae:,.0f} pernoctaciones")
print(f"RMSE: {rmse:,.0f}")
print(f"MAPE: {mape:.2f}%")
print(f"R2:   {r2:.4f}\n")

# 7. VISUALIZACIÓN
fechas_test  = df.index[-MESES_TEST:]
fechas_train = df.index[:-MESES_TEST]

plt.figure(figsize=(14, 5))
plt.plot(fechas_train, y_abs[:-MESES_TEST], label='Datos históricos (Train)', color='#1f77b4', alpha=0.5)
plt.plot(fechas_test, y_test_abs, label='Realidad (Test)', color='green', linewidth=2)
plt.plot(fechas_test, pred_abs, label='Predicción Red Neuronal (LSTM)', color='magenta', linestyle='--', linewidth=2)
plt.axvline(x=fechas_test[0], color='black', linestyle=':', label='Inicio de Test')
plt.title('Validación Ex-Post: Red Neuronal Recurrente (LSTM)', fontsize=14)
plt.ylabel('Pernoctaciones')
plt.xlabel('Fecha')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("lstm_h1_ipt_final.png", dpi=110)
plt.show()

# Curva de aprendizaje (específica de redes neuronales)
fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(historia_final.history["loss"],     label="Train loss", color="#1f77b4")
ax2.plot(historia_final.history["val_loss"], label="Val loss",   color="orange")
ax2.set_title("Curva de aprendizaje LSTM (afinado final)", fontsize=13)
ax2.set_xlabel("Época")
ax2.set_ylabel("MSE (espacio escalado)")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("lstm_h1_curva_aprendizaje.png", dpi=110)
plt.show()

# 8. ÍNDICE DE PRESIÓN TURÍSTICA (IPT) - 3 NIVELES DE ALERTA
# ==============================================================================
# Mismos umbrales (P33/P66 del histórico completo) que en XGBoost, Random
# Forest, Regresión Lineal y SARIMA — los cinco modelos son comparables.
UMBRAL_VERDE_NARANJA, UMBRAL_NARANJA_ROJO = np.percentile(y_abs, [33, 66])
NIVELES_IPT = {0: "Verde", 1: "Naranja", 2: "Rojo"}


def clasificar_ipt(valor_pernoctaciones):
    """Devuelve (nivel_texto, codigo_numerico) donde 0=Verde (menos grave)
    y 2=Rojo (más grave)."""
    if valor_pernoctaciones < UMBRAL_VERDE_NARANJA:
        codigo = 0
    elif valor_pernoctaciones < UMBRAL_NARANJA_ROJO:
        codigo = 1
    else:
        codigo = 2
    return NIVELES_IPT[codigo], codigo


# 8.1 Alertas IPT sobre todo el conjunto de test
alertas_test = pd.DataFrame({
    "fecha_objetivo":        fechas_test,
    "pernoctaciones_real":   y_test_abs,
    "pernoctaciones_predicha": pred_abs,
})
alertas_test[["ipt_real", "codigo_real"]] = alertas_test["pernoctaciones_real"].apply(
    lambda v: pd.Series(clasificar_ipt(v))
)
alertas_test[["ipt_predicho", "codigo_predicho"]] = alertas_test["pernoctaciones_predicha"].apply(
    lambda v: pd.Series(clasificar_ipt(v))
)
alertas_test["acierto_nivel"] = alertas_test["codigo_real"] == alertas_test["codigo_predicho"]

precision_semaforo = alertas_test["acierto_nivel"].mean() * 100
print("=" * 45)
print(" PRECISIÓN DE LA CLASIFICACIÓN SEMÁFORO (IPT)")
print("=" * 45)
print(f"Umbral Verde/Naranja: {UMBRAL_VERDE_NARANJA:,.0f} pernoctaciones")
print(f"Umbral Naranja/Rojo:  {UMBRAL_NARANJA_ROJO:,.0f} pernoctaciones")
print(f"Aciertos de nivel de alerta: {precision_semaforo:.1f}% ({alertas_test['acierto_nivel'].sum()}/{len(alertas_test)} meses)\n")
print(alertas_test.to_string(index=False))

# 8.2 Matriz de confusión entre niveles IPT
matriz_confusion = pd.crosstab(
    alertas_test["ipt_real"], alertas_test["ipt_predicho"],
    rownames=["Real"], colnames=["Predicho"]
).reindex(index=["Verde", "Naranja", "Rojo"], columns=["Verde", "Naranja", "Rojo"], fill_value=0)
print("\nMatriz de confusión (niveles IPT):")
print(matriz_confusion)

# 9. EXPORTAR RESULTADOS
resumen_metrico = pd.DataFrame([{
    "Modelo": arq_str,
    "Horizonte (meses)": 1,
    "MAE": mae, "RMSE": rmse, "MAPE (%)": mape, "R2": r2,
    "Precision_semaforo (%)": precision_semaforo,
}])
resumen_metrico.to_csv("resultados_lstm_ipt_h1.csv", index=False)
alertas_test.to_csv("alertas_ipt_h1_test_lstm.csv", index=False)
matriz_confusion.to_csv("matriz_confusion_ipt_h1_lstm.csv")

print("\nArchivos exportados: resultados_lstm_ipt_h1.csv, alertas_ipt_h1_test_lstm.csv, "
      "matriz_confusion_ipt_h1_lstm.csv, lstm_h1_ipt_final.png, lstm_h1_curva_aprendizaje.png")
